# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bivo2004/my-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule & Reason Codes

**The Rule:** A simple heuristic prioritizing pages that are both stale (not updated in 180+ days) and historically highly visible, ranking them by their 90-day impression volume.

**Reason Codes:**
* `STALE_HIGH_VISIBILITY`: The page is older than 180 days and has >= 500 impressions. High priority for a refresh.
* `STALE_LOW_VISIBILITY`: The page is old but has low impressions. Lower priority.
* `FRESH`: The page was updated recently (under 180 days).

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Setup Colab Environment
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    if not os.getcwd().endswith("flyrank-ml-internship-starter"):
        os.chdir("flyrank-ml-internship-starter")

# 2. Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Data loaded: {df.shape[0]} rows ready for baseline scoring.")

Data loaded: 30000 rows ready for baseline scoring.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked Queue Generation
We will score the pages using our rule, assign the reason codes, and export the top results to a CSV in the `work/outputs/` directory.

In [2]:
# 1. Define the rule components
stale_mask = df["days_since_last_update"] >= 180
visible_mask = df["impressions_90d"] >= 500

# 2. Assign Reason Codes
df["reason_code"] = "FRESH"
df.loc[stale_mask & ~visible_mask, "reason_code"] = "STALE_LOW_VISIBILITY"
df.loc[stale_mask & visible_mask, "reason_code"] = "STALE_HIGH_VISIBILITY"

# 3. Calculate baseline score (rank high-visibility stale pages first)
df["baseline_score"] = stale_mask.astype(int) * visible_mask.astype(int) * df["impressions_90d"]

# 4. Sort and create the queue
queue = df.sort_values("baseline_score", ascending=False).copy()

# 5. Write to CSV
os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print(f"Queue generated and saved to {out_path}")
print(f"Top score in the queue: {queue['baseline_score'].iloc[0]}")

Queue generated and saved to work/outputs/baseline_action_score.csv
Top score in the queue: 61678


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

* **Action:** Flag for editorial review to update content, refresh formatting, and check keyword alignment.
* **Confidence Note:** High confidence that these pages *used* to drive traffic, but low confidence that staleness is the *only* reason they might be dropping.
* **What would make it wrong?** The page might be a seasonal post (e.g., "Summer 2025 Trends") where updating it won't magically bring back winter traffic. Alternatively, a highly visible page might be doing perfectly fine, resulting in a false positive (wasted editorial time).

In [3]:
# Display the Top 20 for review
top_20 = queue[["content_id", "baseline_score", "reason_code", "impressions_90d", "days_since_last_update", "trend_direction"]].head(20)

# Calculate how many of our top 20 are actually declining (Precision@20)
actual_declining = (top_20["trend_direction"] == "down").sum()

print(f"Top 20 Review - Precision@20: {actual_declining}/20 ({(actual_declining/20):.1%}) actually declining.")
display(top_20)

Top 20 Review - Precision@20: 17/20 (85.0%) actually declining.


,content_id,baseline_score,reason_code,impressions_90d,days_since_last_update,trend_direction
16751,content_cf56e2e2e282,61678,STALE_HIGH_VISIBILITY,61678,194,down
16514,content_7368877ea310,59472,STALE_HIGH_VISIBILITY,59472,194,down
7021,content_1bfaa38ff26c,25715,STALE_HIGH_VISIBILITY,25715,194,down
21268,content_0a91db491d14,13299,STALE_HIGH_VISIBILITY,13299,193,down
11489,content_5feee3994adb,7812,STALE_HIGH_VISIBILITY,7812,194,down
12045,content_c2d929d83eaa,7558,STALE_HIGH_VISIBILITY,7558,193,down
698,content_b16bd7307b39,4590,STALE_HIGH_VISIBILITY,4590,194,down
5327,content_fe16a55cd13d,4556,STALE_HIGH_VISIBILITY,4556,194,down
26810,content_ecb6215e79fd,4429,STALE_HIGH_VISIBILITY,4429,194,down
20837,content_928af3e22c80,1697,STALE_HIGH_VISIBILITY,1697,193,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Leakage Check

* **Why the rule is weak:** Sorting purely by `impressions_90d` pushes massive pages to the top, even if their traffic is perfectly `stable` or `up`. This heuristic wastes editorial time on healthy mega-pages while completely ignoring smaller, niche pages that are actively crashing.
* **Leakage Check:** Confirmed. We only used `days_since_last_update` and `impressions_90d`. We did not use `trend_pct` or any future-window labels to calculate the score.

In [4]:
# Verify no leaky columns were used in scoring
print("Checking for leakage in baseline score logic...")
leaky_cols = ["trend_pct", "is_declining", "trend_direction"]
for col in leaky_cols:
    assert col not in df["baseline_score"].name, f"Leakage detected: {col}"

print("Leakage Check: PASS. Only pre-decision signals used.\n")

# Show an example of a "weak pick" (a false positive in the top 20)
weak_picks = top_20[top_20["trend_direction"] != "down"]
print(f"Found {len(weak_picks)} weak picks (false positives) in the Top 20.")
print("These are pages the rule flagged to rewrite, but they aren't actually declining:")

if len(weak_picks) > 0:
    display(weak_picks.head(3))

Checking for leakage in baseline score logic...
Leakage Check: PASS. Only pre-decision signals used.

Found 3 weak picks (false positives) in the Top 20.
These are pages the rule flagged to rewrite, but they aren't actually declining:


,content_id,baseline_score,reason_code,impressions_90d,days_since_last_update,trend_direction
23215,content_bdbec75c1148,1316,STALE_HIGH_VISIBILITY,1316,194,stable
19998,content_e3393b0b5359,0,FRESH,457,13,stable
20000,content_ccaae106ecb6,0,FRESH,43654,104,stable


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.